# Validar modelo `challenger`
- Si pasa la validaciones cambia el alias de: @challenger a @champion
- Adiciona tag: Approval_Check = Approved | Failed

**Job:** Usualmente este notebook es ejecutado como parte de un job automatico. Pasos del job
1. Evaluacion: 
2. Aprobacion: base_parameters={"approval_tag_name": "Approval_Check"}
3. Despliegue: base_parameters={"smoke_test": False}

Parametros del job:
- name="model_full_name", default=medallion_dev.gold.customer_churn_classifier
- name="model_version", default=1

## 1. Configuracion

In [0]:
import sys
python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
print(f"Versión de Python actual: {python_version}")

Versión de Python actual: 3.12.3


In [0]:
import os
current_directory = os.getcwd()
# REQUIRED: Create notebooks for each task
evaluation_notebook_path = f"{current_directory}/03Validation_Churn"
approval_notebook_path = f"{current_directory}/04Approval_Churn" # OPTIONNAL only if Human in the Loop is required
deployment_notebook_path = f"{current_directory}/05Deploy_Churn"

In [0]:
import os
import mlflow
from mlflow.tracking.client import MlflowClient

catalog = "medallion_dev"
schema = "gold"
model_name = "customer_churn_classifier"

dbutils.widgets.text("model_full_name", f"{catalog}.{schema}.{model_name}", "Modelo nombre completo: catalog.schema.name") 
dbutils.widgets.text("model_version", "1", "Modelo Version (@challenger)")

model_full_name = dbutils.widgets.get("model_full_name")
model_version = int(dbutils.widgets.get("model_version"))
model_alias_challenger = "challenger"
model_alias_champion = "champion" 
delete_other_aliases = True

client = MlflowClient()
model_details = client.get_model_version_by_alias(model_full_name, model_alias_challenger)

if model_details:
    model_version = int(model_details.version)
    model_uri = f"models:/{model_full_name}@{model_alias_challenger}"
else:
    print(f"El modelo {model_full_name} no tiene alias {model_alias_challenger}.")
    model_details = client.get_model_version(model_full_name, model_version)
    model_uri = f"models:/{model_full_name}/{model_version}"

if not model_details:
    raise Exception("No encontrado version del modelo registrado en UC.")

requirements_path = mlflow.artifacts.download_artifacts(artifact_uri=f"runs:/{model_details.run_id}/model/requirements.txt") 

if os.path.exists(requirements_path):
    print(f"Se encontró archivo requirements.txt en {requirements_path}")
    with open(requirements_path, 'r') as f:
        requirements_content = f.read()
    print(requirements_content)
else:
    print(f"No se encontró archivo requirements.txt en {requirements_path}")
    requirements_path = mlflow.artifacts.download_artifacts(artifact_uri=f"{model_uri}/model/requirements.txt") 
    if os.path.exists(requirements_path):
        print(f"Opcion 2. Se encontró archivo requirements.txt en {requirements_path}")
    else:
        print(f"Opcion 2. No se encontró archivo requirements.txt en {requirements_path}")

Se encontró archivo requirements.txt en /local_disk0/user_tmp_data/spark-01bc5eb3-2625-449a-a1e3-f3/tmpsh2qk6xq/model/requirements.txt
mlflow==3.12.0
cloudpickle==3.0.0
scipy==1.17.1
scikit-learn==1.8.0
lightgbm==4.6.0
xgboost==3.2.0


In [0]:
# Instalar paquetes desde el requirements.txt del modelo
%pip install --quiet -r $requirements_path

# Instalar dependencia adicional de Databricks
%pip install --quiet databricks-feature-engineering

dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
catalog = "medallion_dev"
schema = "gold"
model_name = "customer_churn_classifier"

dbutils.widgets.text("model_full_name", f"{catalog}.{schema}.{model_name}", "Modelo nombre completo: catalog.schema.name") 
dbutils.widgets.text("model_version", "1", "Modelo Version (@challenger)")

In [0]:
model_full_name = dbutils.widgets.get("model_full_name")
model_version = int(dbutils.widgets.get("model_version"))
catalog = model_full_name.split('.')[0]
schema = model_full_name.split('.')[1]
model_name = model_full_name.split('.')[-1]
feature_table_name = f"{catalog}.{schema}.churn_user_features"

target_col = "churn"
metric_score = "f1"
model_alias_challenger = "challenger" 
model_alias_champion = "champion"
delete_other_aliases = True
rng_seed=42

In [0]:
import mlflow
from mlflow.tracking.client import MlflowClient

client = MlflowClient()
model_details = client.get_model_version_by_alias(model_full_name, model_alias_challenger)

if model_details:
    model_version = int(model_details.version)
    model_uri = f"models:/{model_full_name}@{model_alias_challenger}"
else:
    print(f"El modelo {model_full_name} no tiene alias {model_alias_challenger}.")
    model_details = client.get_model_version(model_full_name, model_version)
    model_uri = f"models:/{model_full_name}/{model_version}"

# run_info = client.get_run(run_id=model_details.run_id)

print(f"Validando modelo: {model_full_name}, version: {model_version}, alias: {model_alias_challenger}")

Validando modelo: medallion_dev.gold.customer_churn_classifier, version: 1, alias: Challenger


## 2. Chequeo predicción

**NOTA**: Puede tomar ~7min porque estamos creando ambiente virtual (env_manager=`virtualenv`) para ejecutar la inferencia y no el serverless cluster del ambiente por defecto.

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql.types import StructType
import pandas as pd

fe = FeatureEngineeringClient()

def spark_udf_predict(df, model_alias: str, result_type: str, useEnvironment=False, env_manager="local"):
    """Aplica modelo a un DataFrame Spark 
    Args:
        df (DataFrame): DataFrame Spark
        model_alias (str): Alias del modelo
        result_type (str): Tipo de resultado
        useEnvironment (bool): Usar entornos
        env_manager (str): ['local', 'conda', 'virtualenv', 'uv']
          'virtualenv': Muy demorado porque tiene que instala: Python y muchos paquetes
          'conda': Usar entornos conda. Databricks Serverless no soporta env_manager = 'conda'
          'uv': Usar entornos universales. Usa Pyton, pero tiene que instalar muchos paquetes
          'local': Usar entornos locales. Recomendado para Databricks Serverless No requiere instalar paquetes
    Returns:
        DataFrame: DataFrame Spark con columna de predicciones
    """

    if useEnvironment:
      predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{model_full_name}@{model_alias}", result_type=result_type, env_manager=env_manager)
    else:
      predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{model_full_name}@{model_alias}", result_type=result_type)

    return df.withColumn('prediction', predict_udf(*predict_udf.metadata.get_input_schema().input_names()))

def batch_predict(df, model_alias: str, result_type: str, useEnvironment=False, env_manager="local"):
    """Aplica modelo a un DataFrame Spark (Presento errores)
    Args:
        df (DataFrame): DataFrame Spark
        model_alias (str): Alias del modelo
        result_type (str): Tipo de resultado
        useEnvironment (bool): Usar entornos
        env_manager (str): ['local', 'conda', 'virtualenv', 'uv']
          'virtualenv': Muy demorado porque tiene que instala: Python y muchos paquetes
          'conda': Usar entornos conda. Databricks Serverless no soporta env_manager = 'conda'
          'uv': Usar entornos universales. Usa Pyton, pero tiene que instalar muchos paquetes
          'local': Usar entornos locales. Recomendado para Databricks Serverless No requiere instalar paquetes
    Returns:
        DataFrame: DataFrame Spark con columna de predicciones
    """
    if useEnvironment:
        predict_df = fe.score_batch(model_uri=f"models:/{model_full_name}@{model_alias}", df=df, result_type=result_type, env_manager=env_manager)
    else:
        predict_df = fe.score_batch(model_uri=f"models:/{model_full_name}@{model_alias}", df=df, result_type=result_type)
    return predict_df

try:
  supported_cols = ["gender", "last_transaction", "canal", "event_count", "days_since_creation", "country", "session_count",
    "order_count", "days_last_event", "days_since_last_activity", "total_amount", "total_item", "age_group", "platform"]

  df = spark.table(feature_table_name).sample(fraction=0.01, seed=rng_seed).limit(100)
  result_type=df.schema[target_col].dataType
  df = df.select(*supported_cols, target_col)
  
  #predict_df = batch_predict(df, model_alias_challenger, result_type, True, "local")
  predict_df = spark_udf_predict(df, model_alias_challenger, result_type, True, "local")

  display(predict_df.limit(5))
          
  predicts_check = True
  print(f"El modelo puede realizar predicciones.")
except Exception as e:
  print(f"No es posible realizar predicción. Error: {e}")
  predicts_check = False

client.set_model_version_tag(name=model_full_name, version=str(model_version), key="predicts", value=predicts_check)

2026/05/08 04:50:06 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2026/05/08 04:50:06 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


gender,last_transaction,canal,event_count,days_since_creation,country,session_count,order_count,days_last_event,days_since_last_activity,total_amount,total_item,age_group,platform,churn,prediction
1,2023-06-08T21:50:54.000Z,WEBAPP,2,1489,SPAIN,1,2,1038,1041,103,5,5,ios,0,0
1,2023-06-09T13:46:13.000Z,WEBAPP,4,1452,SPAIN,4,4,1041,1037,188,9,1,ios,1,1
1,2023-06-07T20:32:59.000Z,PHONE,4,1556,FR,4,4,1036,1043,112,6,4,ios,1,1
1,2023-06-09T05:07:00.000Z,WEBAPP,1,1083,SPAIN,0,1,1041,1044,24,2,3,ios,0,0
1,2023-06-09T15:52:20.000Z,WEBAPP,3,1558,SPAIN,3,3,1037,1036,152,6,3,ios,1,0


El modelo puede realizar predicciones.


In [0]:
from pyspark.sql.types import StructType, IntegerType
import mlflow
import pandas as pd

try:
  supported_cols = ["gender", "last_transaction", "canal", "event_count", "days_since_creation", "country", "session_count",
    "order_count", "days_last_event", "days_since_last_activity", "total_amount", "total_item", "age_group", "platform"]
  
  df = spark.table(feature_table_name).sample(fraction=0.05, seed=rng_seed).select(*supported_cols, target_col).limit(100)
  pdf = df.toPandas()
  model = mlflow.pyfunc.load_model(model_uri)
  predictions = model.predict(pdf)
  pdf['prediction'] = predictions
  
  display(pdf.head(5))
  predicts_check = True
  print(f"El modelo puede realizar predicciones.")
except Exception as e:
  print(f"No es posible realizar predicción. Error: {e}")
  #import traceback
  #traceback.print_exc()
  predicts_check = False

client.set_model_version_tag(name=model_full_name, version=str(model_version), key="predicts", value=predicts_check)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-01bc5eb3-2625-449a-a1e3-f3219641af60/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


gender,last_transaction,canal,event_count,days_since_creation,country,session_count,order_count,days_last_event,days_since_last_activity,total_amount,total_item,age_group,platform,prediction
1,2023-06-08T21:50:54.000Z,WEBAPP,2,1489,SPAIN,1,2,1038,1041,103,5,5,ios,0
1,2023-06-07T21:38:21.000Z,PHONE,4,1228,SPAIN,4,4,1040,1044,110,6,8,ios,1
1,2023-06-08T11:20:45.000Z,WEBAPP,4,1599,USA,3,4,1038,1044,225,9,8,ios,1
1,2023-06-07T04:42:09.000Z,MOBILE,2,1625,USA,2,2,1044,1040,92,3,2,ios,1
1,2023-06-06T20:19:26.000Z,WEBAPP,3,1477,FR,3,3,1036,1044,155,5,8,ios,0


El modelo puede realizar predicciones.


## 3. Chequeo descripcion

In [0]:
if not model_details.description:
  has_description = False
  print("Por favor adicionar descripcion al modelo")
elif not len(model_details.description) > 20:
  has_description = False
  print("Por favor adicionar descripcion al modelo (minimo 20 caracteres).")
else:
  has_description = True
  print(f'El modelo tiene descripcion')

client.set_model_version_tag(name=model_full_name, version=str(model_details.version), key="has_description", value=has_description)

El modelo tiene descripcion


## 4. Chequear artefactos

In [0]:
import os
import shutil

local_dir = "/tmp/model_artifacts"
if not os.path.exists(local_dir):
    os.mkdir(local_dir)

local_path = mlflow.artifacts.download_artifacts(run_id=model_details.run_id, dst_path=local_dir)

if not os.listdir(local_path):
  has_artifacts = False
  print("No hay artefactos asociados con el modelo. MLflow soporta archivos como HTML, .png.")
else:
  has_artifacts = True
  print("El modelo tiene artefactos")
  print("Artefactos descagados en: {}".format(local_path))
  print("Artefactos: {}".format(os.listdir(local_path)))

if os.path.exists(local_dir):
    shutil.rmtree(local_dir)

client.set_model_version_tag(name=model_full_name, version=model_version, key="has_artifacts", value=has_artifacts)

El modelo tiene artefactos
Artefactos descagados en: /tmp/model_artifacts/
Artefactos: ['optuna_parallel_coordinate.png', 'outlier_detection.png', 'confusion_matrix.png', 'feature_target_relationships.png', 'pairwise_relationships.png', 'optuna_slice.png', 'optuna_history.png', 'feature_distributions.png', 'correlation_heatmap.png', 'optuna_importances.png', 'ROC_curve.png', 'target_distributions.png', 'compare_scores.png']


## 5. Metricas de desempeño del modelo

Tipicamente comparar las metricas del modelo `@challenger` con las del modelo `@champion`. Porque ya tenemos registrado un modelo `@champion`, solo recuperamos las metricas del modelo `@champion` sin hacer comparacion.

El modelo registrado captura información de los experimentos MLflow, registradas durante el entrenamiento. Esto da trazabilidad para desplegar el modelo hacia atrás hacia la ejecución de entrenamiento.

Utilizamos una metricas para comparar.

In [0]:
model_run_id = model_details.run_id
score = mlflow.get_run(model_run_id).data.metrics[metric_score]

try:
    champion_model = client.get_model_version_by_alias(model_name, model_alias_champion)
    champion_score = mlflow.get_run(champion_model.run_id).data.metrics[metric_score]
    print(f'{model_alias_champion} {metric_score} score: {champion_score}. {model_alias_challenger} {metric_score} score: {score}.')
    metric_passed = score >= champion_score
except:
    print(f"No encontrado modelo {model_alias_challenger}. Aceptar el modelo, ya que es el primero.")
    metric_passed = True

print(f'Metricas pasadas del modelo: {metric_passed}')
client.set_model_version_tag(name=model_full_name, version=model_details.version, key="metric_passed", value=metric_passed)

No encontrado modelo Champion. Aceptar el modelo, ya que es el primero..
Metricas pasadas del modelo: True


## 6. Metrica de referencia (benchmark) o de negocio en el conjunto de datos de evaluación
Uso del conjunto de validación para chequear el impacto del nuevo modelo.

**NOTA:** Esto evalua el modelo, pero no lo confunda con pruebas A/B. Pruebas A/B es hecho en línea, dividiendo el trafico en dos modelos. El requiere ciclo de retroalimentación para evaluar el efecto de la predicción.

In [0]:
import pandas as pd
import plotly.express as px
from sklearn.metrics import confusion_matrix

# Note: This is over-simplified and depends on the use-case, but the idea is to evaluate our model against business metrics
cost_of_discount = 1 
cost_of_customer_churn = 4 # Por ejemplo 4, implica que el costo de perder un cliente es 4 veces el descuento

cost_true_negative = 0 # No abandono (churn), porque no le dimos el descuento
cost_false_negative = cost_of_customer_churn # Abandono de cliente (churn), perdimos al cliente.
cost_true_positive = cost_of_customer_churn - cost_of_discount # Evitamos el abandono (churn) porque le dimos el descuento
cost_false_positive = -cost_of_discount # No abandono (churn), le dimos el descuento gratis.

def get_model_value(df, model_alias):
    predict_df = spark_udf_predict(df, model_alias, result_type, True, "local")
    pdf = predict_df.toPandas()
    tn, fp, fn, tp = confusion_matrix(pdf[target_col], pdf['prediction']).ravel()
    value = (tn * cost_true_negative) + (fp * cost_false_positive) + (fn * cost_false_negative) + (tp * cost_true_positive)
    #print(f'True Negative: {tn}, False Positive: {fp}, False Negative: {fn}, True Positive: {tp}')
    return float(value)

is_champ_model_exist = True
try:
    champion_model = client.get_model_version_by_alias(model_full_name, model_alias_champion)
    champion_potential_revenue_gain = get_model_value(df, model_alias_champion)
except Exception as e:
    print("Ocurrio un error: ", type(e).__name__, f"Significa que todavía no existe el modelo {model_alias_champion}")
    is_champ_model_exist = False
    champion_potential_revenue_gain = 0

try:
    challenger_potential_revenue_gain = get_model_value(df, model_alias_challenger)
    business_metric_passed = challenger_potential_revenue_gain >= champion_potential_revenue_gain
except Exception as e:
    print("Ocurrio un error al evaluar las metricas del modelo {model_alias_challenger}: ", type(e).__name__)
    challenger_potential_revenue_gain = 0
    print("Saltando evaluacion de metricas de negocio.")
    business_metric_passed = True

print(f'Metrica del negocio benckmark pasada: {business_metric_passed}')
client.set_model_version_tag(name=model_full_name, version=model_details.version, 
                             key="business_metric_passed", value=business_metric_passed)

data = {'Model Alias': [model_alias_challenger, model_alias_champion],
        'Aumento potencial ingresos': [challenger_potential_revenue_gain, champion_potential_revenue_gain]}   

fig = px.bar(data, x='Model Alias', y='Aumento potencial ingresos', color='Model Alias',
    labels={'Aumento potencial ingresos': 'Ingresos impactados'},
    title='Metricas de negocio - Impacto en ingresos')
fig.update_layout(width=600, height=500)
fig.show()

Ocurrio un error:  RestException Significa que todavía no existe el modelo champion


2026/05/08 05:24:15 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2026/05/08 05:24:15 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


Metrica del negocio benckmark pasada: True


## 6. Revisar resultados
Estos son ejemplo de algunos chequeos simples

In [0]:
results = client.get_model_version(model_full_name, model_version)
results.tags

{'best_score': '0.8644',
 'business_metric_passed': 'True',
 'has_artifacts': 'True',
 'has_description': 'True',
 'metric_passed': 'True',
 'predicts': 'True'}

## 7. Promover la nueva versión challenger a champion
Configurar el alias a `@champion`. Si existe alguno, el alias en el modelo `champion` anterior se desactivará automáticamente.

En la practica es recomendado tener separado catalogos para Dev, QA y Prod. La promoción a producción puede ser realizada programaticamente cuando el modelo este listo para promover a producción.

In [0]:
if metric_passed and has_artifacts and has_description and predicts_check and business_metric_passed:
  print(f"Modelo Aprobado: {model_full_name}, Version: {model_details.version}, cambio el alias por: {model_alias_champion}!")
  
  if delete_other_aliases:
    try:
      aliases = model_details.aliases
      for alias in aliases:
          if alias.lower() != model_alias_champion.lower():
              client.delete_registered_model_alias(name=model_full_name, alias=alias)
    except:
      pass
  
  client.set_registered_model_alias(name=model_full_name, alias=model_alias_champion, version=model_details.version)
  client.set_model_version_tag(name=model_full_name, version=model_details.version, key="Approval_Check", value="Approved")  
else:
  print(f"Modelo NO Aprobado: {model_full_name}, Version: {model_details.version}, no fue modificado el alias")
  client.set_model_version_tag(name=model_full_name, version=model_details.version, key="Approval_Check", value="Failed")

Modelo Aprobado: medallion_dev.gold.customer_churn_classifier, Version: 1, cambio el alias: Champion!
